# Non-Comparison Sorts

Comparison-based sorts (merge, quick, heap) have a lower bound of **O(n log n)**.

Non-comparison sorts break this barrier by exploiting properties of the data (e.g., integer range).

| Algorithm | Time | Space | Stable | Constraint |
|-----------|------|-------|--------|------------|
| Counting Sort | O(n + k) | O(n + k) | Yes | k = range of values |
| Radix Sort | O(d × (n + b)) | O(n + b) | Yes | d = digits, b = base |

# Counting Sort

Comparison sorts cannot beat O(n log n) because each comparison yields only one bit of
information. Counting sort sidesteps that entirely: it never compares two elements. Instead
it uses the **value itself as an index**, which is only possible when values are integers in
a known, small range.

**Time:** O(n + k) where k = max value &nbsp; **Space:** O(n + k) &nbsp; **Stable:** yes

## Steps
1. Count the frequency of each value
2. Turn counts into prefix sums - `count[x]` becomes "how many elements are ≤ x", which
   *is* the end position for value x
3. Walk the input **backwards**, placing each element at `count[x] - 1` and decrementing

![Counting Sort Steps](images/counting-sort-steps.png)

## Why is it Stable?

Because of that backwards walk. After the prefix sums, `count[x] - 1` is the *last* position
available to value x, so the last occurrence in the input is placed rightmost, the
second-to-last just before it, and so on - relative order survives.

```
arr = [4, 2, 2, 8, 3, 3, 1]      the prefix sums put the 3s at indices 3 and 4

walking backwards:
  3 → count[3] 5→4 → output[4] = 3      the *later* 3 takes the higher slot
  3 → count[3] 4→3 → output[3] = 3      the earlier 3 lands just before it
```

Iterating forwards instead would reverse equal elements - and radix sort, which leans on
this subroutine being stable, would silently produce wrong answers.

In [ ]:
def counting_sort(arr):
    """
    Stable counting sort for non-negative integers.
    Time: O(n + k), Space: O(n + k) where k = max(arr)
    """
    if not arr:
        return arr
    k = max(arr)
    count = [0] * (k + 1)
    for x in arr:
        count[x] += 1

    # prefix sum - count[i] = number of elements <= i
    for i in range(1, k + 1):
        count[i] += count[i - 1]

    # build output in reverse for stability
    output = [0] * len(arr)
    for x in reversed(arr):
        count[x] -= 1
        output[count[x]] = x
    return output

def test_counting_sort():
    assert counting_sort([4, 2, 2, 8, 3, 3, 1]) == [1, 2, 2, 3, 3, 4, 8]
    assert counting_sort([1, 1, 1]) == [1, 1, 1]
    assert counting_sort([]) == []
    assert counting_sort([5]) == [5]

test_counting_sort()

# Radix Sort

Counting sort needs a small value range, which fails for something like `[170, 45, 802]` --
k would be 802. Radix sort fixes that by sorting one **digit** at a time: each pass only
ever deals with 10 possible values, no matter how large the numbers are.

**Why least significant digit first?** Because each pass is stable, the order established
by earlier (less significant) passes survives whenever the current digits tie. That is
what makes the digit passes compose into a full sort - and it is why the subroutine
*must* be a stable sort.

```
[170, 45, 75, 90, 802, 24, 2, 66]

by 1s     [170, 90, 802, 2, 24, 45, 75, 66]
by 10s    [802, 2, 24, 45, 66, 170, 75, 90]
by 100s   [2, 24, 45, 66, 75, 90, 170, 802]
```

Watch the 45/75 pair on the 10s pass: both have digit 4 and 7 respectively, but in the
100s pass both have digit 0 and tie - stability keeps them in the order the 10s pass
established.

`_counting_sort_by_digit` is the counting sort above with `(x // exp) % 10` extracting the
digit, and a fixed count array of size 10.

**Time:** O(d × (n + 10)) for d digits &nbsp; **Space:** O(n) &nbsp; **Stable:** yes

In [ ]:
def _counting_sort_by_digit(arr, exp):
    """Stable counting sort on a specific digit position (exp = 1, 10, 100, ...)."""
    n = len(arr)
    output = [0] * n
    count = [0] * 10  # base 10 digits

    for x in arr:
        digit = (x // exp) % 10
        count[digit] += 1

    for i in range(1, 10):
        count[i] += count[i - 1]

    for x in reversed(arr):
        digit = (x // exp) % 10
        count[digit] -= 1
        output[count[digit]] = x

    arr[:] = output

def radix_sort(arr):
    """
    LSD radix sort for non-negative integers.
    Time: O(d × (n + 10)), Space: O(n)
    """
    if not arr:
        return arr
    max_val = max(arr)
    exp = 1
    while max_val // exp > 0:
        _counting_sort_by_digit(arr, exp)
        exp *= 10
    return arr

def test_radix_sort():
    assert radix_sort([170, 45, 75, 90, 802, 24, 2, 66]) == [2, 24, 45, 66, 75, 90, 170, 802]
    assert radix_sort([3, 1, 4, 1, 5, 9]) == [1, 1, 3, 4, 5, 9]
    assert radix_sort([]) == []

test_radix_sort()